# Time Series Analysis — Steps

1. Import libraries
2. Load dataset
3. Inspect dataset
4. Filter to relevant airline
5. Clean column names
6. Feature engineering (Year, Month)
7. Rename columns
8. Select features and encode categoricals
9. Define target variable
10. Split data
11. Train Random Forest model
12. Evaluate model


In [1]:
import numpy as np
import pandas as pd

## 1 — Import libraries

Import essential libraries for data processing and modeling.


In [2]:
df=pd.read_excel('OTP_Time_Series_Master_Current_5.xlsx')

## 2 — Load dataset

Load the dataset from the provided Excel file into a DataFrame `df`.


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20263 entries, 0 to 20262
Data columns (total 15 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Route                   20263 non-null  str           
 1   Departing Port          20263 non-null  str           
 2   Arriving Port           20263 non-null  str           
 3   Airline                 20263 non-null  str           
 4   Month                   20263 non-null  datetime64[us]
 5   Sectors Scheduled       20263 non-null  int64         
 6   Sectors Flown           20263 non-null  int64         
 7   Cancellations           20263 non-null  int64         
 8   Departures On Time      20263 non-null  int64         
 9   Arrivals On Time        20263 non-null  int64         
 10  Departures Delayed      20263 non-null  int64         
 11  Arrivals Delayed        20263 non-null  int64         
 12  OnTime Departures 
(%)  20239 non-null  float64       
 1

## 3 — Inspect dataset

Quick inspection to check structure and basic info about the DataFrame.


In [4]:
df = df[df['Airline'].isin([
    "Virgin Australia",
    "Virgin Australia Regional Airlines"
])]

## 4 — Filter dataset

Filter to only include rows for Virgin Australia and its regional airline.


In [5]:
len(df)

4886

## 5 — Check missing values

Verify missing values and dataset length after filtering.


In [6]:
df.isna().sum()

Route                      0
Departing Port             0
Arriving Port              0
Airline                    0
Month                      0
Sectors Scheduled          0
Sectors Flown              0
Cancellations              0
Departures On Time         0
Arrivals On Time           0
Departures Delayed         0
Arrivals Delayed           0
OnTime Departures \n(%)    0
OnTime Arrivals \n(%)      0
Cancellations \n\n(%)      0
dtype: int64

## 6 — Missing values summary

Show per-column missing value counts to decide imputation or removal strategies.


In [7]:
df.info()

<class 'pandas.DataFrame'>
Index: 4886 entries, 262 to 20143
Data columns (total 15 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Route                   4886 non-null   str           
 1   Departing Port          4886 non-null   str           
 2   Arriving Port           4886 non-null   str           
 3   Airline                 4886 non-null   str           
 4   Month                   4886 non-null   datetime64[us]
 5   Sectors Scheduled       4886 non-null   int64         
 6   Sectors Flown           4886 non-null   int64         
 7   Cancellations           4886 non-null   int64         
 8   Departures On Time      4886 non-null   int64         
 9   Arrivals On Time        4886 non-null   int64         
 10  Departures Delayed      4886 non-null   int64         
 11  Arrivals Delayed        4886 non-null   int64         
 12  OnTime Departures 
(%)  4886 non-null   float64       
 13  O

## 7 — Re-check dataset info

Confirm DataFrame shape and column types after cleaning steps.


In [8]:
df.columns = (
    df.columns
      .str.replace(r'\s+', ' ', regex=True)
      .str.strip()
)

print(df.columns)

Index(['Route', 'Departing Port', 'Arriving Port', 'Airline', 'Month',
       'Sectors Scheduled', 'Sectors Flown', 'Cancellations',
       'Departures On Time', 'Arrivals On Time', 'Departures Delayed',
       'Arrivals Delayed', 'OnTime Departures (%)', 'OnTime Arrivals (%)',
       'Cancellations (%)'],
      dtype='str')


## 8 — Clean column names

Normalize column names by removing extra whitespace and trimming.


In [9]:
df['Year'] = df['Month'].dt.year
df['Month_Num'] = df['Month'].dt.month

## 9 — Feature engineering

Extract `Year` and `Month_Num` from the `Month` datetime column for modeling.


In [10]:
df.rename(
    columns={'OnTime Arrivals\n(%)': 'OnTimeArrivalRate'},
    inplace=True
)

## 10 — Rename columns

Rename verbose or awkward column names for clarity before modeling.


In [11]:
X = df[
    [
        'Route',
        'Departing Port',
        'Arriving Port',
        'Airline',
        'Sectors Scheduled',
        'Year',
        'Month_Num'
    ]
]

## 11 — Select features

Choose the predictors to use for modeling and store them in `X`.


In [12]:
X = pd.get_dummies(X, drop_first=True)

## 12 — Encode categorical variables

Convert categorical features into numeric dummies for modeling (`drop_first=True`).


In [13]:
y = df.iloc[:,-4]

## 13 — Define target variable

Set the target variable `y` (OnTimeArrivalRate or appropriate column).


In [14]:
print(y)

262      77.777778
263      77.205882
264      73.684211
265      90.476190
266      83.271375
           ...    
20139    75.000000
20140    80.000000
20141    75.000000
20142    75.000000
20143    77.519380
Name: OnTime Arrivals (%), Length: 4886, dtype: float64


## 14 — Preview target

Display the `y` series to confirm correct selection.


In [15]:
# lets build regression model Random forest regressor\
from sklearn.model_selection import train_test_split

## 15 — Modeling: prepare pipeline

Import model helpers and set up modeling pipeline (train/test split follows).


In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
)

## 16 — Split data

Split `X` and `y` into training and testing sets (`test_size=0.2`).


In [17]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

## 17 — Train model

Instantiate and train a `RandomForestRegressor` on the training data.


In [18]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))


MAE: 7.497104355517319
MSE: 101.53042080473635
R2: 0.5691401554587713


## 18 — Evaluate model

Compute MAE, MSE, and R2 to assess model performance on the test set.
